In [17]:
import os
import numpy as np
import fitsio
from astropy.io import fits
from astropy.table import Table
from tqdm import tqdm
import math
import pandas as pd

In [58]:
# First test on 2020 SGA data
sga2020 = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', hdu=1)
sga2020_RA_DEC = Table()
sga2020_RA_DEC['TargetID'] = sga2020['SGA_ID']
sga2020_RA_DEC['Target_RA'] = sga2020['RA']
sga2020_RA_DEC['Target_DEC'] = sga2020['DEC']
sga2020_RA_DEC.sort('Target_RA')
df2020 = sga2020_RA_DEC.to_pandas()
#df2020[0:1000].to_csv("/pscratch/sd/q/qshimp/SGA2020-data/sga2020_RA_DEC_test4.csv")
df2020

,TargetID,Target_RA,Target_DEC
0,495237,0.000466,31.731567
1,942428,0.000639,-45.954255
2,1431578,0.000691,-44.578090
3,435797,0.001160,-41.422890
4,859355,0.002361,32.209246
...,...,...,...
383615,423345,359.997059,8.565006
383616,494567,359.997564,24.907425
383617,692705,359.997696,21.192246
383618,43806,359.998960,-49.918054


In [67]:
# Check for duplicate Target IDs
duplicates = df2020[df2020.duplicated(subset='TargetID', keep=False)]
len(duplicates)

0

In [78]:
# Check for duplicate Target RAs for varying degrees of rounding
for decimals in [10, 9, 8]:
    temp = df2020.copy()
    temp['RA_round'] = temp['Target_RA'].round(decimals)

    duplicates = temp[temp.duplicated(subset='RA_round', keep=False)]

    print(f"{decimals} decimals: {len(duplicates)} duplicate rows")

    print(temp)

10 decimals: 0 duplicate rows
        TargetID   Target_RA  Target_DEC    RA_round
0         495237    0.000466   31.731567    0.000466
1         942428    0.000639  -45.954255    0.000639
2        1431578    0.000691  -44.578090    0.000691
3         435797    0.001160  -41.422890    0.001160
4         859355    0.002361   32.209246    0.002361
...          ...         ...         ...         ...
383615    423345  359.997059    8.565006  359.997059
383616    494567  359.997564   24.907425  359.997564
383617    692705  359.997696   21.192246  359.997696
383618     43806  359.998960  -49.918054  359.998960
383619   1238372  359.999529   -5.282050  359.999529

[383620 rows x 4 columns]
9 decimals: 0 duplicate rows
        TargetID   Target_RA  Target_DEC    RA_round
0         495237    0.000466   31.731567    0.000466
1         942428    0.000639  -45.954255    0.000639
2        1431578    0.000691  -44.578090    0.000691
3         435797    0.001160  -41.422890    0.001160
4         859

In [79]:
# Check for duplicate Target Decs for varying degrees of rounding
for decimals in [10, 9, 8]:
    temp = df2020.copy()
    temp['DEC_round'] = temp['Target_DEC'].round(decimals)

    duplicates = temp[temp.duplicated(subset='DEC_round', keep=False)]

    print(f"{decimals} decimals: {len(duplicates)} duplicate rows")

    print(temp)

10 decimals: 0 duplicate rows
        TargetID   Target_RA  Target_DEC  DEC_round
0         495237    0.000466   31.731567  31.731567
1         942428    0.000639  -45.954255 -45.954255
2        1431578    0.000691  -44.578090 -44.578090
3         435797    0.001160  -41.422890 -41.422890
4         859355    0.002361   32.209246  32.209246
...          ...         ...         ...        ...
383615    423345  359.997059    8.565006   8.565006
383616    494567  359.997564   24.907425  24.907425
383617    692705  359.997696   21.192246  21.192246
383618     43806  359.998960  -49.918054 -49.918054
383619   1238372  359.999529   -5.282050  -5.282050

[383620 rows x 4 columns]
9 decimals: 2 duplicate rows
        TargetID   Target_RA  Target_DEC  DEC_round
0         495237    0.000466   31.731567  31.731567
1         942428    0.000639  -45.954255 -45.954255
2        1431578    0.000691  -44.578090 -44.578090
3         435797    0.001160  -41.422890 -41.422890
4         859355    0.002361  

In [83]:
for decimals in range(10, 0, -1):
    temp = df2020.copy()

    temp['RA_round'] = temp['Target_RA'].round(decimals)
    temp['DEC_round'] = temp['Target_DEC'].round(decimals)

    duplicates = temp[
        temp.duplicated(
            subset=['RA_round', 'DEC_round'],
            keep=False
        )
    ]

    if len(duplicates) > 0:
        print(f"Coordinate dilution occurs at {decimals} decimals")
        print(duplicates.sort_values(['RA_round', 'DEC_round']))
        df2020['RA_round'] = df2020['Target_RA'].round(decimals + 1)
        df2020['DEC_round'] = df2020['Target_DEC'].round(decimals + 1)
        break
duplicates = df2020[df2020.duplicated(subset=['Target_RA', 'Target_DEC'], keep=False)]
len(duplicates)
df2020

Coordinate dilution occurs at 3 decimals
        TargetID   Target_RA  Target_DEC  RA_round  DEC_round
94137    1425732   71.317968   -2.820086    71.318     -2.820
94138      90016   71.318130   -2.819668    71.318     -2.820
100172   1394035   78.370226  -51.324605    78.370    -51.325
100173    784230   78.370342  -51.324758    78.370    -51.325
133214   1424895  129.045169   -3.433813   129.045     -3.434
133215    353804  129.045333   -3.434159   129.045     -3.434
150735   1030619  142.950637   55.742641   142.951     55.743
150738   1377789  142.951371   55.743402   142.951     55.743
150819   1386084  143.018606   12.154354   143.019     12.154
150821   1218660  143.018940   12.153742   143.019     12.154
163558    977469  152.566860   54.987515   152.567     54.988
163562    391707  152.567499   54.988360   152.567     54.988
179920    757148  162.605716   -0.336407   162.606     -0.336
179923     99410  162.606431   -0.336131   162.606     -0.336
202358     84595  175.317917 

,TargetID,Target_RA,Target_DEC,RA_round,DEC_round
0,495237,0.000466,31.731567,0.0005,31.7316
1,942428,0.000639,-45.954255,0.0006,-45.9543
2,1431578,0.000691,-44.578090,0.0007,-44.5781
3,435797,0.001160,-41.422890,0.0012,-41.4229
4,859355,0.002361,32.209246,0.0024,32.2092
...,...,...,...,...,...
383615,423345,359.997059,8.565006,359.9971,8.5650
383616,494567,359.997564,24.907425,359.9976,24.9074
383617,692705,359.997696,21.192246,359.9977,21.1922
383618,43806,359.998960,-49.918054,359.9990,-49.9181


In [52]:
# Copy code to run fp_cutouts in galaxy morphology directory 
# python FP_cutouts.py --csv /pscratch/sd/q/qshimp/SGA2025-data/sga2025_RA_DEC_scale=1.5.csv --outdir /pscratch/sd/q/qshimp/Cutouts/sga2025/Scale_1.5

In [48]:
sgaBeta2025 = Table.read('/global/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/SGA2025-beta-parent-refcat-v1.6.kd.fits', hdu = 1)
sgaBeta2025.sort('diam', reverse=True)
sgaBeta2025

ra,dec,ref_id,mag,fitmode,pa,ba,diam
float64,float64,int64,float32,int32,float32,float32,float32
80.894167,-69.756111,5053785,0.802,6,171.0,0.85,645.65
13.186667,-72.828611,5053799,2.747,6,45.0,0.64,380.19
143.8079,-36.6991,5053749,10.74,6,156.0,0.62,151.2
23.462041666666703,30.66022222222221,4931495,6.351,2,22.7,0.59,77.60863
177.31,-18.413,5053766,12.15,6,0.0,1.0,62.4
209.3,26.8,5053754,12.6,6,99.0,0.65,56.0
39.95830000000001,-34.4997,5053771,9.032,2,42.7,0.7,47.76
11.892520663592299,-25.289267919714405,4971304,7.94,0,50.94034,0.15635367,43.514473
15.0183,-33.7186,5053794,10.368,2,92.0,0.68,39.810005


In [55]:
PIX_SCALE       = 0.262       # arcsec/pixel for Legacy Surveys
CUTOUT_SIZE_PIX = 152         # fixed 152x152 cutouts
scale = 1 # Manually adjust this to change allowed galaxy size
diamMax = scale*PIX_SCALE*CUTOUT_SIZE_PIX/60 # max diameter in arcmin
print(str(diamMax) + " arcmin")

sizedTargets = sgaBeta2025[sgaBeta2025['diam'] <= diamMax]
print(len(sizedTargets))
bigs = sgaBeta2025[sgaBeta2025['diam'] > diamMax]
bigs
# Mistakenly used 39.824 arcmin instead of arsec; this pulled 8 largest objects
# First 3 have no galaxy, fourth and fifth are Magellanic Clouds, sixth is Sculptor Galaxy, 
# seventh is a faint, large cloud of stars, eighth is Triangulum Galaxy

0.6637333333333333 arcmin
216313


ra,dec,ref_id,mag,fitmode,pa,ba,diam
float64,float64,int64,float32,int32,float32,float32,float32
80.894167,-69.756111,5053785,0.802,6,171.0,0.85,645.65
13.186667,-72.828611,5053799,2.747,6,45.0,0.64,380.19
143.8079,-36.6991,5053749,10.74,6,156.0,0.62,151.2
23.462041666666703,30.66022222222221,4931495,6.351,2,22.7,0.59,77.60863
177.31,-18.413,5053766,12.15,6,0.0,1.0,62.4
209.3,26.8,5053754,12.6,6,99.0,0.65,56.0
39.95830000000001,-34.4997,5053771,9.032,2,42.7,0.7,47.76
11.892520663592299,-25.289267919714405,4971304,7.94,0,50.94034,0.15635367,43.514473
15.0183,-33.7186,5053794,10.368,2,92.0,0.68,39.810005


In [ ]:
for decimals in range(10, 0, -1):
    temp = df2025.copy()

    temp['RA_round'] = temp['Target_RA'].round(decimals)
    temp['DEC_round'] = temp['Target_DEC'].round(decimals)

    duplicates = temp[
        temp.duplicated(
            subset=['RA_round', 'DEC_round'],
            keep=False
        )
    ]

    if len(duplicates) > 0:
        print(f"Coordinate dilution occurs at {decimals} decimals")
        print(duplicates.sort_values(['RA_round', 'DEC_round']))
        df['RA_round'] = df['Target_RA'].round(decimals + 1)
        df['DEC_round'] = df['Target_DEC'].round(decimals + 1)
        break
duplicates = df[df.duplicated(subset=['Target_RA', 'Target_DEC'], keep=False)]
len(duplicates)
df

In [54]:
# Create table of 2025 SGA data
sga2025_RA_DEC = Table()
sga2025_RA_DEC['TargetID'] = sizedTargets['ref_id']
sga2025_RA_DEC['Target_RA'] = sizedTargets['ra']
sga2025_RA_DEC['Target_DEC'] = sizedTargets['dec']
sga2025_RA_DEC['Diam'] = sizedTargets['diam']
df = sga2025_RA_DEC.to_pandas()
df[:3].to_csv("/pscratch/sd/q/qshimp/SGA2025-data/sga2025_RA_DEC_scale=1.5.csv")
df

,TargetID,Target_RA,Target_DEC,Diam
0,4914454,221.122623,5.176362,0.995596
1,1100,110.621415,-86.587703,0.995578
2,4717758,11.904244,-20.486096,0.995576
3,4130259,191.621516,-33.435353,0.995569
4,4821684,344.729830,-8.457870,0.995568
...,...,...,...,...
380957,5053877,243.445960,54.369076,0.063233
380958,5054141,24.096522,15.964525,0.057806
380959,5053949,156.616452,67.654027,0.054249
380960,5056388,188.996026,70.848279,0.051551
